In [1]:
!pip install xarray

In [2]:
# Step 1: Install and import
!pip install --upgrade xarray zarr gcsfs cftime nc-time-axis
!pip install xarray

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import zarr
import gcsfs

xr.set_options(display_style='html')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.rcParams['figure.figsize'] = 12, 6

# Step 2: Load the catalog
df = pd.read_csv('https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv')
print(df.columns.tolist())
df.head()

['activity_id', 'institution_id', 'source_id', 'experiment_id', 'member_id', 'table_id', 'variable_id', 'grid_label', 'zstore', 'dcpp_init_year', 'version']


,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,zstore,dcpp_init_year,version
0,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,ps,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706
1,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,rsds,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706
2,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,rlus,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706
3,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,rlds,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706
4,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,psl,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706


In [3]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", 
                       "xarray", "zarr", "gcsfs", "cftime", "nc-time-axis"])


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


0

In [5]:
df.head()

,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,zstore,dcpp_init_year,version
0,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,ps,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706
1,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,rsds,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706
2,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,rlus,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706
3,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,rlds,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706
4,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,psl,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,NaN,20170706


In [4]:
# Step 3: Filter to your exact parameters
query = df[
    (df['variable_id'] == 'tas') &          # near-surface air temp
    (df['table_id'] == 'Amon') &            # monthly atmospheric
    (df['experiment_id'].isin(['historical', 'ssp245', 'ssp585'])) &
    (df['member_id'] == 'r1i1p1f1') &
    (df['grid_label'] == 'gr')
]

# Pick one model that has all 3 scenarios available
# MIROC6 and MPI-ESM1-2-LR are reliable choices
query = query[query['source_id'] == 'MPI-ESM1-2-LR']
print(query[['source_id', 'experiment_id', 'zstore']])

Empty DataFrame
Columns: [source_id, experiment_id, zstore]
Index: []


In [6]:
# Check what's available for your parameters
query = df[
    (df['variable_id'] == 'tas') &
    (df['table_id'] == 'Amon') &
    (df['experiment_id'].isin(['historical', 'ssp245', 'ssp585'])) &
    (df['member_id'] == 'r1i1p1f1') &
    (df['grid_label'] == 'gr')
]

print("Total matches:", len(query))
print("\nAvailable models and scenarios:")
print(query.groupby(['source_id', 'experiment_id']).size().unstack(fill_value=0))

Total matches: 32

Available models and scenarios:
experiment_id      historical  ssp245  ssp585
source_id                                    
CIESM                       1       1       1
E3SM-1-0                    1       0       0
E3SM-1-1                    1       1       1
E3SM-1-1-ECA                1       0       0
EC-Earth3                   1       1       1
EC-Earth3-AerChem           1       0       0
EC-Earth3-CC                1       1       1
EC-Earth3-Veg               1       1       1
EC-Earth3-Veg-LR            1       1       1
FGOALS-f3-L                 1       1       1
IPSL-CM5A2-INCA             1       0       0
IPSL-CM6A-LR                1       1       1
IPSL-CM6A-LR-INCA           1       0       0
KACE-1-0-G                  1       1       1


In [7]:
# Filter to IPSL-CM6A-LR with all 3 scenarios
query_filtered = query[query['source_id'] == 'IPSL-CM6A-LR']
print(query_filtered[['source_id', 'experiment_id', 'zstore']])

           source_id experiment_id  \
29231   IPSL-CM6A-LR    historical   
47969   IPSL-CM6A-LR        ssp245   
279556  IPSL-CM6A-LR        ssp585   

                                                   zstore  
29231   gs://cmip6/CMIP6/CMIP/IPSL/IPSL-CM6A-LR/histor...  
47969   gs://cmip6/CMIP6/ScenarioMIP/IPSL/IPSL-CM6A-LR...  
279556  gs://cmip6/CMIP6/ScenarioMIP/IPSL/IPSL-CM6A-LR...  


In [8]:
# Load and preview just the historical dataset first to make sure it works
gcs = gcsfs.GCSFileSystem(token='anon')

row = query_filtered[query_filtered['experiment_id'] == 'historical'].iloc[0]
store = gcs.get_mapper(row['zstore'])
ds = xr.open_zarr(store, consolidated=True)
print(ds)

I0504 13:47:30.317831  764300 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0504 13:47:30.320871  764317 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0504 13:47:30.320920  764317 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0504 13:47:30.320923  764317 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0504 13:47:30.320924  764317 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0504 13:47:30.320926  764317 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0504 13:47:30.320928  764317 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0504 13:47:30.320929  764317 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0504 13:47:30.320930  764317 ev_poll_posix.cc:593] FD from fork parent still in p

<xarray.Dataset> Size: 163MB
Dimensions:      (time: 1980, lat: 143, lon: 144, axis_nbounds: 2)
Coordinates:
  * time         (time) datetime64[ns] 16kB 1850-01-16T12:00:00 ... 2014-12-1...
  * lat          (lat) float32 572B -90.0 -88.73 -87.46 ... 87.46 88.73 90.0
  * lon          (lon) float32 576B 0.0 2.5 5.0 7.5 ... 350.0 352.5 355.0 357.5
    time_bounds  (time, axis_nbounds) datetime64[ns] 32kB ...
    height       float64 8B ...
Dimensions without coordinates: axis_nbounds
Data variables:
    tas          (time, lat, lon) float32 163MB ...
Attributes: (12/52)
    CMIP6_CV_version:       cv=6.2.3.5-2-g63b123e
    Conventions:            CF-1.7 CMIP-6.2
    EXPID:                  historical
    NCO:                    "4.6.0"
    activity_id:            CMIP
    branch_method:          standard
    ...                     ...
    table_id:               Amon
    title:                  IPSL-CM6A-LR model output prepared for CMIP6 / CM...
    tracking_id:            hdl:21.14100/

In [9]:
scenarios = ['historical', 'ssp245', 'ssp585']
results = []

for scenario in scenarios:
    print(f"Loading {scenario}...")
    row = query_filtered[query_filtered['experiment_id'] == scenario].iloc[0]
    store = gcs.get_mapper(row['zstore'])
    ds = xr.open_zarr(store, consolidated=True)
    
    # Area-weighted global mean (accounts for smaller grid cells near poles)
    weights = np.cos(np.deg2rad(ds['lat']))
    tas_weighted = ds['tas'].weighted(weights)
    global_mean = tas_weighted.mean(dim=['lat', 'lon'])
    
    # Resample to annual mean
    print(f"  Computing annual mean...")
    annual = global_mean.resample(time='1Y').mean()
    
    # Convert to dataframe
    df_out = annual.to_dataframe(name='tas').reset_index()
    df_out['tas_celsius'] = df_out['tas'] - 273.15  # Kelvin → Celsius
    df_out['scenario'] = scenario
    df_out['year'] = df_out['time'].dt.year
    results.append(df_out[['year', 'scenario', 'tas_celsius']])
    print(f"  Done! Years: {df_out['year'].min()} - {df_out['year'].max()}")

final_df = pd.concat(results).reset_index(drop=True)
print("\nFinal shape:", final_df.shape)
print(final_df.groupby('scenario')[['year', 'tas_celsius']].agg({'year': ['min','max'], 'tas_celsius': 'mean'}))

Loading historical...
  Computing annual mean...
  Done! Years: 1850 - 2014
Loading ssp245...


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/xarray/groupers.py:543: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  self.index_grouper = pd.Grouper(


  Computing annual mean...
  Done! Years: 2015 - 2100
Loading ssp585...


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/xarray/groupers.py:543: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  self.index_grouper = pd.Grouper(


  Computing annual mean...
  Done! Years: 2015 - 2100

Final shape: (337, 3)
            year       tas_celsius
             min   max        mean
scenario                          
historical  1850  2014   13.142957
ssp245      2015  2100   15.545157
ssp585      2015  2100   16.472559


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/xarray/groupers.py:543: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  self.index_grouper = pd.Grouper(


In [10]:
# Fix the deprecation warning while we're at it
final_df.to_csv('global_temp_scenarios.csv', index=False)

# Preview what the file looks like
print(final_df.head(10))
print("...")
print(final_df.tail(10))

   year    scenario  tas_celsius
0  1850  historical    12.607056
1  1851  historical    12.788422
2  1852  historical    12.777710
3  1853  historical    12.852295
4  1854  historical    12.947266
5  1855  historical    12.858368
6  1856  historical    12.983765
7  1857  historical    12.941956
8  1858  historical    12.878448
9  1859  historical    12.995300
...
     year scenario  tas_celsius
327  2091   ssp585    18.913513
328  2092   ssp585    18.922089
329  2093   ssp585    18.890411
330  2094   ssp585    18.982697
331  2095   ssp585    19.258698
332  2096   ssp585    19.371429
333  2097   ssp585    19.257477
334  2098   ssp585    19.493530
335  2099   ssp585    19.732819
336  2100   ssp585    19.569641
